In [ ]:
import os
from glob import glob
import pandas as pd
import cv2
import numpy as np
import os
from onekey_algo import get_param_in_cwd


def merge_gray_images(output_path, *ipath, reszie=(299, 299)):
    if len(ipath) == 1:
        image_arrs = [cv2.resize(cv2.imread(ip, cv2.IMREAD_GRAYSCALE), reszie) for ip in ipath] * 3
    else:
        image_arrs = [cv2.resize(cv2.imread(ip, cv2.IMREAD_GRAYSCALE), reszie) for ip in ipath]
    merged_image = np.dstack(image_arrs)
    np.save(output_path, merged_image)
    
root = get_param_in_cwd('radio_dir')
files = {}
surrounds = ['', '_+01', '_+02', '_+04', '_-01', '_-02', '_-04']

def get_id(x):
    x = x.replace('.npy', '.gz')
    for p in ['_+01', '_+02', '_+04', '_-01', '_-02', '_-04']:
        if p in x:
            x = x.replace(p, '')
    x = x.replace('.png', '.gz')
    return x

samples = pd.DataFrame(glob(os.path.join(root, '*', 'crop', '*.png')), columns=['fpath'])
samples['ID'] = samples['fpath'].map(lambda x: get_id(os.path.basename(x)))
label_data = pd.read_csv(get_param_in_cwd('label_file'))
samples = pd.merge(samples, label_data, on='ID', how='inner')
samples

In [ ]:
train_data = samples[samples['group'] == 'train']
train_data[['fpath', 'label']].to_csv('split_info/train.txt', index=False, header=False, sep='\t')

test_data = samples[samples['group'] != 'train']
test_data[['fpath', 'label']].to_csv('split_info/val.txt', index=False, header=False, sep='\t')

print(f"Training set: {train_data.shape[0]}({len(np.unique(train_data['ID']))}); test set: {test_data.shape[0]}({len(np.unique(test_data['ID']))})")